## Notebook for preprocessing of Joung2023 dataset

Step 0 (base dataset): Start from TFAtlas_subsample_raw

Step 1 (annotation enrichment), add: TF identity, is_differentiated, is_combinatorial_tested

Step 2 (balancing): QC, cap cells per TF (max 1000, min 1000, keep differentiated as much as possible) 

Step 3 (preprocessing): keep raw counts, normalize + log1p

Step 4 (controls): check whether to merge GFP and mCherry

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

In [2]:

adata_file = '../../../data/real/Joung2023_Giorgi/preprocessed_joung_data.h5ad'
result_folder = '../../../data/real/Joung2023_Giorgi/'

In [3]:
adata = sc.read_h5ad(adata_file)

In [4]:
adata

AnnData object with n_obs × n_vars = 64000 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'condition', 'louvain', 'dataset', 'cell_type', 'intervention', 'set'
    var: 'n_cells-0-double', 'n_cells-1-double', 'n_cells-2-double', 'n_cells-3-double', 'n_cells-4-double', 'n_cells-5-double', 'n_cells-6-double', 'n_cells-7-double', 'n_cells-8-double', 'n_cells-0-double-single', 'n_cells-1-double-single', 'n_cells-2-double-single', 'n_cells-3-double-single', 'n_cells-4-double-single', 'n_cells-5-double-single', 'n_cells-6-double-single', 'n_cells-7-double-single', 'n_cells-8-double-single', 'highly_variable-double-single', 'means-double-single', 'dispersions-double-single', 'dispersions_norm-double-single', 'mean-double-single', 'std-double-single', 'n_cells-0-0-perturbed-single', 'n_cells-0-1-0-perturbed-single', 'n_cells-1-1-0-perturbed-single', 'n_cells-2-1-0-perturbed-single', 'n_cells-3-1-0-perturbed-single', 'n_cells-0-2-0-perturbed-single', 'n_cells-1-2-0-pertur

In [5]:
adata.obs

,n_genes,percent_mito,n_counts,batch,TF,condition,louvain,dataset,cell_type,intervention,set
"R1.21,R2.72,R3.22,P1.62-3-1-perturbed-single",1757,0.019720,2282.0,perturbed,TFORF0098-MSGN1,TFORF0098-MSGN1,NaN,single,unknown,MSGN1,test
"R1.56,R2.96,R3.91,P1.30-1-1-perturbed-single",1662,0.016735,2211.0,perturbed,TFORF0098-MSGN1,TFORF0098-MSGN1,NaN,single,unknown,MSGN1,test
"R1.02,R2.87,R3.94,P1.30-1-1-perturbed-single",1660,0.026340,2202.0,perturbed,TFORF0098-MSGN1,TFORF0098-MSGN1,NaN,single,unknown,MSGN1,test
"R1.48,R2.75,R3.48,P1.62-3-1-perturbed-single",2498,0.017972,3728.0,perturbed,TFORF0098-MSGN1,TFORF0098-MSGN1,NaN,single,unknown,MSGN1,test
"R1.28,R2.02,R3.44,P1.62-3-1-perturbed-single",4042,0.019712,7508.0,perturbed,TFORF0098-MSGN1,TFORF0098-MSGN1,NaN,single,unknown,MSGN1,test
...,...,...,...,...,...,...,...,...,...,...,...
"R1.96,R2.86,R3.20,P1.31-8-double",1240,0.014785,1488.0,8,"TFORF0391-FERD3L,TFORF0098-MSGN1",TFORF0391-FERD3L+TFORF0098-MSGN1,1,double,unknown,FERD3L+MSGN1,test
"R1.96,R2.87,R3.15,P1.31-8-double",1385,0.038373,1746.0,8,"TFORF0391-FERD3L,TFORF0098-MSGN1",TFORF0391-FERD3L+TFORF0098-MSGN1,1,double,unknown,FERD3L+MSGN1,test
"R1.96,R2.94,R3.85,P1.31-8-double",3540,0.010881,6893.0,8,"TFORF1216-NHLH1,TFORF0098-MSGN1",TFORF1216-NHLH1+TFORF0098-MSGN1,3,double,unknown,MSGN1+NHLH1,test
"R1.96,R2.94,R3.94,P1.31-8-double",4196,0.006881,9446.0,8,"TFORF0391-FERD3L,TFORF0098-MSGN1",TFORF0391-FERD3L+TFORF0098-MSGN1,3,double,unknown,FERD3L+MSGN1,test


In [9]:
adata.obs['intervention'].value_counts()

intervention
FERD3L+MSGN1    23171
MSGN1+PTF1A     15878
PTF1A            4000
unperturbed      4000
FERD3L+FLI1      3490
MSGN1+NHLH1      2653
FERD3L           2472
FLI1+PTF1A       1870
MSGN1+NR5A2      1302
MSGN1             959
FLI1+MSGN1        946
FLI1+NHLH1        630
KLF4              325
FERD3L+PAX5       324
FLI1+NR5A2        303
HNF4A+MSGN1       282
FERD3L+KLF4       213
PAX5+PTF1A        173
KLF4+PTF1A        157
MSGN1+PAX5        131
NHLH1             122
FLI1              119
KLF4+MSGN1         97
PAX5               86
NR5A2              74
FLI1+HNF4A         71
NHLH1+PAX5         49
KLF4+NHLH1         36
KLF4+NR5A2         20
HNF4A              19
NR5A2+PAX5         16
HNF4A+PAX5          7
HNF4A+KLF4          5
Name: count, dtype: int64

In [10]:
my_adata = ad.read_h5ad('/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCAL/gCRL/data/real/Joung2023/Joung2023.h5ad')

In [16]:
my_adata.obs_names.isin(adata.obs_names.str.replace('-perturbed-single', '')).sum()/ len(my_adata.obs_names)

np.float64(0.007801144572824422)

In [17]:
adata.obs[adata.obs['intervention'] == 'unperturbed']

,n_genes,percent_mito,n_counts,batch,TF,condition,louvain,dataset,cell_type,intervention,set
"R1.41,R2.50,R3.41,P1.62-3-1-perturbed-single",3774,0.015696,6562.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.37,R2.04,R3.95,P1.22-0-1-perturbed-single",3633,0.020878,6562.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.52,R2.29,R3.19,P1.30-1-1-perturbed-single",2650,0.019377,4077.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.01,R2.65,R3.67,P1.30-1-1-perturbed-single",2119,0.019563,3067.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.39,R2.73,R3.45,P1.62-3-1-perturbed-single",2604,0.026511,4036.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
...,...,...,...,...,...,...,...,...,...,...,...
"R1.96,R2.82,R3.11,P1.22-0-1-perturbed-single",2121,0.021461,3122.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.14,R2.45,R3.10,P1.30-1-1-perturbed-single",1109,0.019956,1353.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.47,R2.48,R3.96,P1.38-2-1-perturbed-single",2020,0.023477,2939.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training
"R1.50,R2.37,R3.43,P1.62-3-1-perturbed-single",2927,0.020838,4751.0,perturbed,TFORF3550-mCherry,ctrl,NaN,single,unknown,unperturbed,training


In [ ]:
adata.obs.loc[adata.obs['dataset'] == 'single', 'intervention'].astype('str').value_counts()

intervention
unperturbed    4000
PTF1A          4000
FERD3L         2472
MSGN1           959
KLF4            325
NHLH1           122
FLI1            119
PAX5             86
NR5A2            74
HNF4A            19
Name: count, dtype: int64

In [22]:
adata.obs.loc[adata.obs['dataset'] == 'double', 'intervention'].astype('str').value_counts()

intervention
FERD3L+MSGN1    23171
MSGN1+PTF1A     15878
FERD3L+FLI1      3490
MSGN1+NHLH1      2653
FLI1+PTF1A       1870
MSGN1+NR5A2      1302
FLI1+MSGN1        946
FLI1+NHLH1        630
FERD3L+PAX5       324
FLI1+NR5A2        303
HNF4A+MSGN1       282
FERD3L+KLF4       213
PAX5+PTF1A        173
KLF4+PTF1A        157
MSGN1+PAX5        131
KLF4+MSGN1         97
FLI1+HNF4A         71
NHLH1+PAX5         49
KLF4+NHLH1         36
KLF4+NR5A2         20
NR5A2+PAX5         16
HNF4A+PAX5          7
HNF4A+KLF4          5
Name: count, dtype: int64